Do the prompts differ in how far away the AI is from the correct number of mistakes?

In [11]:
#For absolute error

import pandas as pd
from scipy.stats import friedmanchisquare
from itertools import combinations
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

# Load data
df = pd.read_csv("Output_extraction/ai_grading_final_v3.csv")

# Clean columns
df["answer_key_id"] = df["answer_key_id"].astype(str).str.strip()
df["prompt_type"] = df["prompt_type"].astype(str).str.strip()

# Create outcome variable
df["absolute_error"] = abs(
    df["ai_estimated_mistakes"] - df["true_mistakes"]
)

# Prompt order
prompt_order = [
    "very_pessimistic",
    "pessimistic",
    "neutral",
    "confident",
    "very_confident"
]

# If answer_key_id is not unique, create a repetition index
df["rep"] = df.groupby(
    ["answer_key_id", "true_mistakes", "prompt_type"]
).cumcount()

# Convert to wide format
df_wide = df.pivot_table(
    index=["answer_key_id", "true_mistakes", "rep"],
    columns="prompt_type",
    values="absolute_error",
    aggfunc="first"
)

# Keep only complete rows with all five prompt types
df_wide = df_wide[prompt_order].dropna()

# Friedman test
friedman_result = friedmanchisquare(
    df_wide["very_pessimistic"],
    df_wide["pessimistic"],
    df_wide["neutral"],
    df_wide["confident"],
    df_wide["very_confident"]
)

print("Friedman test for absolute error")
print("Statistic:", friedman_result.statistic)
print("p-value:", friedman_result.pvalue)

Friedman test for absolute error
Statistic: 36.47473080247425
p-value: 2.3107791272153729e-07


In [13]:
posthoc_results = []

for p1, p2 in combinations(prompt_order, 2):
    test = wilcoxon(
        df_wide[p1],
        df_wide[p2],
        zero_method="wilcox",
        alternative="two-sided"
    )

    posthoc_results.append({
        "comparison": f"{p1} vs {p2}",
        "p_value": test.pvalue
    })

posthoc_df = pd.DataFrame(posthoc_results)

# Holm correction
posthoc_df["p_holm"] = multipletests(
    posthoc_df["p_value"],
    method="holm"
)[1]

print(posthoc_df)

                           comparison       p_value        p_holm
0     very_pessimistic vs pessimistic  2.062233e-11  2.062233e-10
1         very_pessimistic vs neutral  7.700160e-05  5.390112e-04
2       very_pessimistic vs confident  4.113862e-07  3.291090e-06
3  very_pessimistic vs very_confident  6.796063e-11  6.116456e-10
4              pessimistic vs neutral  7.247233e-04  3.623617e-03
5            pessimistic vs confident  1.018886e-01  3.056659e-01
6       pessimistic vs very_confident  7.537136e-01  7.537136e-01
7                neutral vs confident  1.256165e-01  3.056659e-01
8           neutral vs very_confident  3.967104e-04  2.380263e-03
9         confident vs very_confident  5.764918e-02  2.305967e-01


In [14]:
#For signed error

import pandas as pd
from scipy.stats import friedmanchisquare
from itertools import combinations
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

# Load data
df = pd.read_csv("Output_extraction/ai_grading_final_v3.csv")

# Clean columns
df["answer_key_id"] = df["answer_key_id"].astype(str).str.strip()
df["prompt_type"] = df["prompt_type"].astype(str).str.strip()

# Create signed error variable
df["signed_error"] = (
    df["ai_estimated_mistakes"] - df["true_mistakes"]
)

# Prompt order
prompt_order = [
    "very_pessimistic",
    "pessimistic",
    "neutral",
    "confident",
    "very_confident"
]

# If answer_key_id is not unique, create a repetition index
df["rep"] = df.groupby(
    ["answer_key_id", "true_mistakes", "prompt_type"]
).cumcount()

# Convert to wide format
df_wide = df.pivot_table(
    index=["answer_key_id", "true_mistakes", "rep"],
    columns="prompt_type",
    values="signed_error",
    aggfunc="first"
)

# Keep only complete rows with all five prompt types
df_wide = df_wide[prompt_order].dropna()

# Friedman test
friedman_result = friedmanchisquare(
    df_wide["very_pessimistic"],
    df_wide["pessimistic"],
    df_wide["neutral"],
    df_wide["confident"],
    df_wide["very_confident"]
)

print("Friedman test for signed error")
print("Statistic:", friedman_result.statistic)
print("p-value:", friedman_result.pvalue)

Friedman test for signed error
Statistic: 26.117720433456302
p-value: 2.9961198424479565e-05


In [15]:
posthoc_results = []

for p1, p2 in combinations(prompt_order, 2):
    test = wilcoxon(
        df_wide[p1],
        df_wide[p2],
        zero_method="wilcox",
        alternative="two-sided"
    )

    posthoc_results.append({
        "comparison": f"{p1} vs {p2}",
        "p_value": test.pvalue
    })

posthoc_df = pd.DataFrame(posthoc_results)

# Holm correction
posthoc_df["p_holm"] = multipletests(
    posthoc_df["p_value"],
    method="holm"
)[1]

print(posthoc_df)

                           comparison       p_value        p_holm
0     very_pessimistic vs pessimistic  1.284432e-06  8.991024e-06
1         very_pessimistic vs neutral  3.403218e-07  2.722574e-06
2       very_pessimistic vs confident  2.275159e-10  2.275159e-09
3  very_pessimistic vs very_confident  1.217081e-07  1.095373e-06
4              pessimistic vs neutral  5.685056e-01  1.000000e+00
5            pessimistic vs confident  2.814298e-02  1.688579e-01
6       pessimistic vs very_confident  4.159151e-01  1.000000e+00
7                neutral vs confident  1.049374e-01  5.246869e-01
8           neutral vs very_confident  7.397785e-01  1.000000e+00
9         confident vs very_confident  1.769996e-01  7.079983e-01
